# Fine Tuning of VGGFace for Facial Recognition

In [ ]:
import time
import sys
import gdown
import os
from typing import List, cast, Any
from numpy.typing import NDArray
#
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Convolution2D,
                                     ZeroPadding2D,
                                     MaxPooling2D,
                                     Flatten,
                                     Dropout,
                                     Activation)

## Load Base Model
* `base_model()` constructs sequential model
* `load_model()` then applies the weights from the provided url
* `model` the base model with the applied weights

In [ ]:
BASE_MODEL_WEIGHTS_URL = (
    "https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5"
)

In [ ]:
def base_model() -> Sequential:
  model = Sequential()
  model.add(ZeroPadding2D((1, 1), input_shape=(224, 224, 3)))
  model.add(Convolution2D(64, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(64, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(128, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(128, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(256, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(ZeroPadding2D((1, 1)))
  model.add(Convolution2D(512, (3, 3), activation="relu"))
  model.add(MaxPooling2D((2, 2), strides=(2, 2)))
  #
  model.add(Convolution2D(4096, (7, 7), activation="relu"))
  model.add(Dropout(0.5))
  model.add(Convolution2D(4096, (1, 1), activation="relu"))
  model.add(Dropout(0.5))
  model.add(Convolution2D(2622, (1, 1)))
  model.add(Flatten())
  model.add(Activation("softmax"))
  return model

In [ ]:
def load_model(
  url: str = BASE_MODEL_WEIGHTS_URL
) -> Model:
  # construct model
  model: Model = base_model()
  # check if file already downloaded
  def download_weights_if_necessary(
      file_name: str = "base_weights.h5",
      url:str = url
    ) -> os.path:
    target_file = os.path.normpath(f"vggFineTuning/{file_name}")
    if not os.path.isdir('vggFineTuning'):
      os.mkdir('vggFineTuning')
    if os.path.isfile(target_file):
      print(f"weights already downloaded at {target_file}")
      return target_file
    #
    try:
      print(f"{file_name} will be downloaded from {url} to {target_file}")
      gdown.download(BASE_MODEL_WEIGHTS_URL, target_file, quiet=False)
    except Exception as e:
      raise ValueError(
        f"Something has gone while downloading {file_name} from {url}"
      ) from e
    # potentially add downloading/uncompressing of .zip/.bz2 files?
    return target_file
  weight_file = download_weights_if_necessary()
  try:
    model.load_weights(weight_file)
  except Exception as e:
    raise ValueError(
      f"Something has gone loading pre-trained weights from {weight_file}"
    ) from e
  # model here will be 2622d dimensions
  base_model_output = Flatten()(model.layers[-5].output)
  # flatten to 4096 dimensions
  # apparently increases accuracy according to the comments in the DeepFace code
  vgg_face_descriptor = Model(inputs=model.layers[0].input, outputs=base_model_output)
  return vgg_face_descriptor


In [ ]:
try:
  model = load_model(url=BASE_MODEL_WEIGHTS_URL)
  model.summary()
except Exception as e:
  import traceback
  traceback.print_exc()

## Download & Preparing Dataset
[Link to Dataset](https://www.kaggle.com/competitions/11-785-fall-20-homework-2-part-2/data)

In [ ]:
import kagglehub
# API token (do not steal):
#KGAT_4bd8e8f794e814c9efea224d573e0f44
# print("vvv this will appear even if you're already logged in <3")
path = "/kaggle/input/competitions/11-785-fall-20-homework-2-part-2"
# kagglehub.login()
# !kaggle competitions download -c "11-785-fall-20-homework-2-part-2" -p "/kaggle/input/data/"
# path = kagglehub.competition_download("11-785-fall-20-homework-2-part-2")
print(f"dataset input at {path}")
test_dir  = path + "/classification_data/test_data"
train_dir = path + "/classification_data/train_data"
val_dir   = path + "/classification_data/val_data"
print(f"testing data directory: {test_dir}")
print(f"training data directory: {train_dir}")
print(f"validation data directory: {val_dir}")

In [ ]:
from tensorflow.keras.utils import image_dataset_from_directory
#
# model input shape is (None, 224, 224, 3)
# > 224x224 image, 3 colour channels
batch_size = 32
image_size = (224, 224)
#
train_ds = image_dataset_from_directory(
  directory         = train_dir,
  # validation_split  = 0.75, # 0.25 for training
  # subset            = 'training',
  # seed              = 123,
  image_size        = image_size,
  batch_size        = batch_size
)
val_ds = image_dataset_from_directory(
  directory         = val_dir,
  validation_split  = 0.75, # 0.25 for training
  subset            = 'training',
  seed              = 123,
  image_size        = image_size,
  batch_size        = batch_size
)
test_ds = image_dataset_from_directory(
  directory         = test_dir,
  # validation_split  = 0.75, # 0.25 for training
  # subset            = 'training',
  # seed              = 123,
  image_size        = image_size,
  batch_size        = batch_size
)

## Fine Tuning Time
we have a large dataset; therefore we should be able to get full fine tuning.
If I have time may extend to potentially also have the option to only fine tune the final layers

In [ ]:
# checkpoints incase of crash or timeout or anything else :(
from tensorflow.keras.callbacks import ModelCheckpoint
checkpoint_path = "model_checkpoint/cp-{epoch:04d}.weights.h5"
checkpoint_callback = ModelCheckpoint(
    filepath          = checkpoint_path,
    save_weights_only = True,
    monitor           = 'val_accuracy',
    save_freq         = 'epoch'
)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy as Loss_SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalCrossentropy as Metric_SparseCategoricalCrossentropy
model.trainable = True
model.compile(
  optimizer=Adam(1e-5),
  loss=Loss_SparseCategoricalCrossentropy(from_logits=True),
  metrics=[Metric_SparseCategoricalCrossentropy('accuracy')]
)
#
num_epochs = 5
#
history = model.fit(
  val_ds,
  epochs          = num_epochs,
  validation_data = val_ds,
  callbacks       = [checkpoint_callback]
)

In [ ]:
restore_from_checkpoing = False
if restore_from_checkpoing:
  from tensorflow.train import Checkpoint
  checkpoint = Checkpoint(model)
  checkpoint.restore(checkpoint_path)

In [ ]:
if 0 == 1:
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    #
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 8))
    plt.subplot(2, 1, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.ylim([0.8, 1])
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.subplot(2, 1, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.ylim([0, 1.0])
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.show()

## Save Fine Tuned Weights

In [ ]:
model.save_weights(
  'vggFineTuning/new_weights.weights.h5',
  overwrite=True
)